# Ariel 2025 — Notebook 3: Deep Sequence Models

So sánh nhóm **deep learning trên light curve thô**: CNN1D, TCN, LSTM, GRU, Transformer, Autoencoder+MLP.

Khác với Notebook 2 (features dạng bảng), các model này nhận tensor chuỗi `[planets, time, channels]` với channels = FGS white-light + AIRS light curves (bin theo bước sóng). Mỗi model có head `(mu, log_sigma)` và được train bằng Gaussian NLL — xuất σ trực tiếp.

> **Khuyến nghị bật GPU** (Settings → Accelerator → GPU). Deep models dễ overfit khi ít planet — dùng như nhóm baseline so sánh, không nhất thiết là model chính.

In [ ]:
# === Setup: clone running branch and make ariel_ml importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

CANDIDATE_SRC = [
    str(CLONE_DIR / "src"),
    "/kaggle/input/ariel-ml-src/src",
    "/kaggle/usr/lib/ariel_ml",
    "src", "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using ariel_ml from:", _p); break
else:
    print("WARNING: ariel_ml source not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = OUTPUT_DIR / "weights"; WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Cấu hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import DeepModelConfig
from metrics import ariel_naive_reference
from training import evaluate_prediction
from deep_models import (
    CNN1DRegressor, TCNRegressor, LSTMRegressor, GRURegressor,
    TransformerSequenceRegressor, AutoencoderMLPRegressor,
)

LIMIT = 1100
TIME_STEPS = 128
WAVELENGTH_BINS = 64
EPOCHS = 200        # max epochs; early stopping (patience=20) will cut this short
BATCH_SIZE = 16     # smaller batch → more gradient steps per epoch (~55/epoch at 1100 planets)
HIDDEN = 128
RANDOM_STATE = 42

## 2. Load tensor đã precompute

Tensor được build bởi **Notebook 3a** (CPU) và đã push lên branch `running`.  
Đảm bảo các giá trị `LIMIT`, `TIME_STEPS`, `WAVELENGTH_BINS` khớp với lúc build.

In [ ]:
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
data_file = PRECOMPUTED_DIR / f"sequence_train_L{LIMIT}_T{TIME_STEPS}_W{WAVELENGTH_BINS}.npz"
data = np.load(data_file, allow_pickle=True)
Xn  = data["Xn"]
Y   = data["Y"]
tr  = data["tr"]
va  = data["va"]
ids = data["planet_ids"].tolist()
print(f"Loaded: Xn={Xn.shape}, Y={Y.shape}, train={len(tr)}, val={len(va)}")

## 3. Train và Evaluate tất cả kiến trúc

In [ ]:
ARCHITECTURES = {
    "cnn1d": CNN1DRegressor,
    "tcn": TCNRegressor,
    "lstm": LSTMRegressor,
    "gru": GRURegressor,
    "transformer": TransformerSequenceRegressor,
    "autoencoder_mlp": AutoencoderMLPRegressor,
}

import torch
device_str = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device_str)

# Naive reference computed from training targets (same convention as tabular models).
naive_ref = ariel_naive_reference(Y[tr])

rows = []
deep_preds = {}
for name, cls in ARCHITECTURES.items():
    cfg = DeepModelConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hidden_size=HIDDEN,
        random_state=RANDOM_STATE,
        device="auto",
        n_components=30,   # match tabular models; reduces 283-dim output to 30 PCA components
        patience=20,       # early stopping: restore best weights after 20 epochs without val improvement
    )
    model = cls(cfg).fit(Xn[tr], Y[tr], x_val=Xn[va], y_val=Y[va])
    weight_path = WEIGHTS_DIR / f"{name}_weights.pt"
    model.save_weights(weight_path)
    print(f"  → weights saved: {weight_path}")
    pred = model.predict(Xn[va])
    deep_preds[name] = pred
    ev = evaluate_prediction(Y[va], pred, naive_reference=naive_ref)
    rows.append({"model": name, "ariel_gll_score": ev.ariel_gll_score,
                 "rmse_mean": ev.rmse_mean, "gaussian_nll": ev.gaussian_nll,
                 "coverage_1sigma": ev.coverage_1sigma})
    print(f"{name:20s} GLL={ev.ariel_gll_score:.4f}  RMSE={ev.rmse_mean:.3e}  cov1={ev.coverage_1sigma:.3f}")

deep_table = pd.DataFrame(rows).sort_values("ariel_gll_score", ascending=False)
deep_table.to_csv(OUTPUT_DIR / "exp5_deep_sequence.csv", index=False)
deep_table

## 5. Ghi chú cho báo cáo
- Deep sequence models học pattern ingress/egress trực tiếp từ light curve, không cần feature engineering thủ công.
- Nhưng **dễ overfit** khi số planet nhỏ; cần nhiều dữ liệu + regularization + GPU.
- So sánh trực tiếp với nhóm physics-based ML (Notebook 2): thường Bayesian Ridge + PHC ổn định và calibrate σ tốt hơn trên ít dữ liệu, trong khi deep models có thể vượt khi dữ liệu lớn.
- Bảng `exp5_deep_sequence.csv` dùng chung trục metric (Ariel GLL) với các experiment khác để so sánh công bằng.

## 6. So sánh ĐỒNG BỘ ML vs Deep (cùng split + PHC)
Đánh giá ML và Deep trên **đúng tập eval**, **cùng naive_ref (train)**, **cùng PHC per-wavelength** (fit calibrator trên nửa `va`, eval trên nửa còn lại → leak-free). ML chạy trên feature dạng bảng của **cùng planet** (`ids` theo thứ tự npz). Đây là so sánh apples-to-apples; bảng deep ở trên là single-split không PHC.

> §6 **nạp weights deep đã lưu** (`weights/*_weights.pt`) nên KHÔNG cần chạy lại vòng train ở trên — chỉ cần: setup → config → load `.npz` → §6 (attach `weights/` + features).

In [ ]:
import glob
from config import ModelConfig, DeepModelConfig
from estimators import ModelFactory
from metrics import SigmaCalibrator, ariel_naive_reference
from training import evaluate_prediction
from models import ModelPrediction
from deep_models import (CNN1DRegressor, TCNRegressor, LSTMRegressor, GRURegressor,
                         TransformerSequenceRegressor, AutoencoderMLPRegressor)

# --- Nạp dự đoán deep từ WEIGHTS đã lưu (KHÔNG train lại) ---
DEEP_ARCH = {"cnn1d": CNN1DRegressor, "tcn": TCNRegressor, "lstm": LSTMRegressor,
             "gru": GRURegressor, "transformer": TransformerSequenceRegressor,
             "autoencoder_mlp": AutoencoderMLPRegressor}
def _find_weight(name):
    cands = [WEIGHTS_DIR / f"{name}_weights.pt"] + [Path(q) for q in glob.glob(f"/kaggle/input/**/{name}_weights.pt", recursive=True)]
    return next((c for c in cands if Path(c).exists()), None)

deep_preds = {}
for name, cls in DEEP_ARCH.items():
    wp = _find_weight(name)
    if wp is None:
        print("thiếu weights, skip:", name); continue
    cfg = DeepModelConfig(hidden_size=HIDDEN, n_components=30, random_state=RANDOM_STATE, device="auto")
    m = cls(cfg).load_weights(wp, n_time=Xn.shape[1], n_channels=Xn.shape[2], y_train=Y[tr])  # refit PCA từ Y[tr]
    deep_preds[name] = m.predict(Xn[va])
    print("loaded:", name, "<-", wp)

# --- Feature dạng bảng cho CÙNG planet (thứ tự ids của npz) ---
cands = (glob.glob(str(PRECOMPUTED_DIR / "features_train_*.csv"))
         + glob.glob(str(OUTPUT_DIR / "features_train*.csv"))
         + glob.glob("/kaggle/input/**/features_train*.csv", recursive=True))
assert cands, "Không thấy features_train_*.csv — attach output prepare_features."
feats = pd.read_csv(cands[0]); feats["planet_id"] = feats["planet_id"].astype(str)
print("features:", cands[0], feats.shape)
Xtab_all = feats.set_index("planet_id").reindex([str(i) for i in ids])
fcols = [c for c in Xtab_all.columns if pd.api.types.is_numeric_dtype(Xtab_all[c])]
Xtab = Xtab_all[fcols].fillna(0.0).to_numpy(float)

# --- chia va -> cal/eval (model CHƯA thấy va -> leak-free) ---
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(va)); half = len(va) // 2
cal_pos, eval_pos = perm[:half], perm[half:]
Yva = Y[va]; naive_ref = ariel_naive_reference(Y[tr])

def phc_eval(mu, sig):
    sig = np.maximum(np.asarray(sig, float), 1e-12)
    cal = SigmaCalibrator(per_target=True).fit(Yva[cal_pos], mu[cal_pos], sig[cal_pos])
    sig_eval = cal.transform(sig[eval_pos])
    return evaluate_prediction(Yva[eval_pos], ModelPrediction(mu[eval_pos], sig_eval), naive_reference=naive_ref)

rows = []
for name, pred in deep_preds.items():                       # deep + PHC
    ev = phc_eval(np.asarray(pred.mu, float), np.asarray(pred.sigma, float))
    rows.append({"model": name, "type": "deep+PHC", "ariel_gll_score": ev.ariel_gll_score,
                 "rmse_mean": ev.rmse_mean, "coverage_1sigma": ev.coverage_1sigma})
for name in ["bayesian_ridge", "extra_trees", "random_forest"]:   # ML cùng split + PHC
    m = ModelFactory.create(name, ModelConfig(n_components=30, random_state=RANDOM_STATE, calibrate_sigma=False))
    m.fit(Xtab[tr], Y[tr])
    p = m.predict(Xtab[va])
    ev = phc_eval(np.asarray(p.mu, float), np.asarray(p.sigma, float))
    rows.append({"model": name, "type": "ml+PHC", "ariel_gll_score": ev.ariel_gll_score,
                 "rmse_mean": ev.rmse_mean, "coverage_1sigma": ev.coverage_1sigma})

sync = pd.DataFrame(rows).sort_values("ariel_gll_score", ascending=False).reset_index(drop=True)
sync.to_csv(OUTPUT_DIR / "sync_ml_vs_deep.csv", index=False)
print("=== Apples-to-apples (cùng eval split + PHC) ===")
print(sync.to_string(index=False))

mm = sync.sort_values("ariel_gll_score")
colors = ["#d9534f" if "deep" in t else "#4f86d9" for t in mm["type"]]
plt.figure(figsize=(8, max(3, 0.5 * len(mm))))
plt.barh(mm["model"], mm["ariel_gll_score"], color=colors)
plt.xlabel("Ariel GLL (same eval split + PHC)"); plt.title("Synchronized ML vs Deep")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "sync_ml_vs_deep.png", dpi=150, bbox_inches="tight"); plt.show()
